# 作业 3.1：从标准源能谱提取 HPGe 探测器性能

## 标准源与测量能谱

Eurica gamma 探测阵列由 12 个 Euroball Cluster 组成，每个 Cluster 含 7 个 HPGe 晶体；标定源距探测器约 22 cm。数据已对每个 Cluster 的 7 个晶体作 add-back，再合并 12 个 Cluster，用于考察阵列的整体性能。装置详情见 [Installation and commissioning of EURICA – Euroball-RIKEN Cluster Array](https://www.sciencedirect.com/science/article/pii/S0168583X13003182)。

<img src="eurica.png" alt="Eurica detector array" style="max-width:36%;" />

能谱由 $^{152}$Eu 与 $^{133}$Ba 标准源测得，测量开始于 2013 年 2 月 13 日，记录时长为 7442 s。两个源的参考日期均为 1998 年 1 月 1 日；参考活度分别为 $^{152}$Eu 40.9 kBq（5%）和 $^{133}$Ba 42.2 kBq（3%）。活度衰变修正采用 $T_{1/2}(^{152}\mathrm{Eu})=13.517$ y、$T_{1/2}(^{133}\mathrm{Ba})=3849.3$ d。

<div class="source-line-grid">
<table>
<thead><tr><th>Nuclide</th><th><i>E</i><sub>γ</sub> (keV)</th><th><i>P</i><sub>γ</sub> (%)</th></tr></thead>
<tbody>
<tr><td><sup>133</sup>Ba</td><td>80.9979</td><td>34.06</td></tr>
<tr><td><sup>152</sup>Eu</td><td>121.7817</td><td>28.41</td></tr>
<tr><td><sup>152</sup>Eu</td><td>244.6974</td><td>7.55</td></tr>
<tr><td><sup>133</sup>Ba</td><td>276.3989</td><td>7.164</td></tr>
<tr><td><sup>133</sup>Ba</td><td>302.8508</td><td>18.33</td></tr>
<tr><td><sup>152</sup>Eu</td><td>344.2785</td><td>26.59</td></tr>
</tbody>
</table>
<table>
<thead><tr><th>Nuclide</th><th><i>E</i><sub>γ</sub> (keV)</th><th><i>P</i><sub>γ</sub> (%)</th></tr></thead>
<tbody>
<tr><td><sup>133</sup>Ba</td><td>356.0129</td><td>62.05</td></tr>
<tr><td><sup>152</sup>Eu</td><td>778.9045</td><td>12.93</td></tr>
<tr><td><sup>152</sup>Eu</td><td>867.378</td><td>4.23</td></tr>
<tr><td><sup>152</sup>Eu</td><td>964.079</td><td>14.51</td></tr>
<tr><td><sup>152</sup>Eu</td><td>1112.076</td><td>13.67</td></tr>
<tr><td><sup>152</sup>Eu</td><td>1408.013</td><td>20.87</td></tr>
</tbody>
</table>
</div>

<style>
.source-line-grid { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:1rem; align-items:start; }
.source-line-grid table { width:100%; margin:0; }
.source-line-grid th:nth-child(n+2), .source-line-grid td:nth-child(n+2) { text-align:right; }
@media (max-width:720px) { .source-line-grid { grid-template-columns:1fr; } }
</style>

能量、$P_\gamma$ 和半衰期以推荐核数据为准，可查阅 [DDEP/LNHB recommended decay data](https://www.lnhb.fr/home/nuclear-data/)。$P_\gamma$ 是每次母核衰变发射该 gamma ray 的概率，不是源活度；它可辅助指认谱线，但不能直接当作实验 peak area 之比。

本作业使用 [gamma.root](gamma.root) 中的 `TH1F h0`。横轴尚未刻度，单位为 channel；每个 bin 宽 0.2 channel。下面是本次测量的合并能谱。纵轴采用 log scale，使强峰和弱峰能够同时显示。

<img src="standard_source_spectrum.png" alt="measured Eu-152 and Ba-133 spectrum" style="max-width:62%;" />

对照标准源的参考能谱与上表，可以先辨认较强的 gamma line，再利用初步线性刻度寻找其余 peak。

<div style="display:flex; flex-wrap:wrap; gap:1rem; align-items:center;">
  <img src="../calibration_method/152Eu.png" alt="Eu-152 reference spectrum" style="max-width:34%; height:auto;" />
  <img src="../calibration_method/133Ba.png" alt="Ba-133 reference spectrum" style="max-width:37%; height:auto;" />
</div>


本作业从标准源测得的 gamma 能谱中提取 HPGe 探测器的三个性能参数：

| 探测器性能 | 从 gamma peak 中提取的量 |
| --- | --- |
| 能量刻度 | centroid $\mu_{ch}$ 与已知的 $E_\gamma$ |
| 能量分辨率 | Gaussian width $\sigma_{ch}$ |
| full-energy peak efficiency | 扣除本底后的 peak area $N_{\mathrm{peak}}$ |

三项分析都从选定 gamma peak 的局部拟合开始。

## Gamma peak 与局部本底

感兴趣的 gamma peak 叠加在连续本底上。孤立、近似对称的 peak 可先用 Gaussian signal 描述：

$$
s(x)=H\exp\!\left[-\frac{(x-\mu)^2}{2\sigma^2}\right],
$$

其中 $H$ 是 peak height，$\mu$ 是 centroid，$\sigma$ 是 Gaussian width。在足够窄的拟合区间内，连续本底可先作 linear approximation：

$$
b(x)=b_0+b_1(x-x_0),\qquad f(x)=s(x)+b(x).
$$

这里的 linear 只描述 peak 附近的局部变化，并不表示完整 gamma spectrum 的本底是直线。实际本底可能包含 Compton continuum 或 edge、邻近 peak、低能 tail 和电子学响应。

本例先在 visible peak 之外选择左右 sideband，单独拟合一条直线，用它确定 $b_0$ 和 $b_1$ 的初值；随后在完整局部区间内同时拟合 Gaussian 与 linear background。第二步不固定本底参数，因此得到的是 joint fit，其 covariance matrix 同时反映 peak 与本底参数的相关性。

867.378 keV peak 的 peak-to-background ratio 适合展示这一过程。左图给出 sideband 上的 linear fit；右图上半部分采用 linear y，显示 joint fit 的 signal + background 和最终 background，下半部分给出 Pearson residual。

<div class="peak-example-grid">
  <figure>
    <img src="fit_867_background.png" alt="linear fit to sidebands near the 867.378 keV peak" />
    <figcaption>左右 sideband 给出 linear background 的初值。</figcaption>
  </figure>
  <figure>
    <img src="fit_867_linear.png" alt="867.378 keV gamma peak with a linear background" />
    <figcaption>Gaussian + linear background 的 joint fit 与 residual。</figcaption>
  </figure>
</div>

<style>
.peak-example-grid { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:1rem; max-width:78%; margin:1rem 0; align-items:start; }
.peak-example-grid figure { margin:0; }
.peak-example-grid img { width:100%; height:auto; }
.peak-example-grid figcaption { margin-top:.4rem; font-size:.92rem; line-height:1.35; }
@media (max-width:720px) { .peak-example-grid { grid-template-columns:1fr; max-width:100%; } }
</style>

### 进阶讨论：本底模型并不唯一

如果 peak 前后的本底 level 明显不同，可以尝试在 linear term 上加入

$$
\frac{S}{2}\operatorname{erfc}\!\left(\frac{x-\mu}{\sqrt{2}\sigma}\right)
$$

来描述一种经过能量分辨平滑的 step。但增加参数必须带来可见的 residual 改善；在本数据对 1408 keV peak 的试算中，`erfc` 改变了本底的过渡形状，却没有实质改善 residual，因此不作为本作业的一般模型。

[ORTEC GammaVision 用户手册](https://www.ortec-online.com/-/media/ametekortec/manuals/a/a66-mnl.pdf?la=en)以 straight-line background 作为基本 ROI/singlet 处理，并在谱形需要时采用 stepped 或 parabolic background。[肖石良等（2024）](https://wulixb.iphy.ac.cn/pdf-content/10.7498/aps.73.20231980.pdf)对更复杂的在线 gamma spectrum 分别加入低能 tail、`erfc` step、polynomial background 和 Compton-edge response。这说明 `erfc` 也只描述本底中的一种结构，不能解释所有形状。


### 拟合参数、peak area 与误差

本作业的数据是 histogram 计数，实例使用 binned Poisson likelihood。拟合后除检查 fit status，还要看 residual 是否在零附近无规则分布：

$$
r_i=\frac{n_i-\nu_i}{\sqrt{\nu_i}},
$$

其中 $n_i$ 是第 $i$ 个 bin 的观测计数，$\nu_i$ 是模型给出的期望计数。

ROOT 的 `gaus` 参数 $H$ 是 peak height，不是 area。若 histogram 的 bin width 为 $w$，Gaussian signal 的总计数为

$$
N_{\mathrm{peak}}=\frac{H\sigma\sqrt{2\pi}}{w}.
$$

`GetParError` 可直接读取 $\mu$ 和 $\sigma$ 的拟合误差。area 同时依赖 $H$ 和 $\sigma$，两者来自包含本底参数的同一次 joint fit，通常存在相关性。由拟合结果的 covariance matrix 得到

$$
u_N^2=\left(\frac{\sqrt{2\pi}}{w}\right)^2
\left[\sigma^2u_H^2+H^2u_\sigma^2+
2H\sigma\operatorname{Cov}(H,\sigma)\right].
$$

拟合本身会给出参数 covariance matrix。常见误区发生在拟合之后：只读取各参数的 `GetParError`，然后把参数当作彼此独立。对于由多个 fitted parameters 得到的量 $g(\mathbf p)$，必须使用

$$
u_g^2=\nabla g^{\mathsf T}\,C\,\nabla g,
$$

其中 $C$ 是同一次拟合返回的 covariance matrix；只保留对角项等于假设所有参数不相关。对于本底平滑的孤立 peak，还可以用 sideband subtraction 对 fitted area 作简单交叉检查。


## 从 peak 参数得到探测器性能

### 能量刻度

用各参考线的 centroid $ch_i$ 与已知能量 $E_i$ 拟合

$$E(ch)=a_0+a_1ch.$$

先由两个相隔较远、指认可靠的 peak 建立粗略线性关系，再据此定位其余参考线。最终用 calibration residual

$$\Delta E_i=E_{i,\mathrm{ref}}-E_{\mathrm{cal}}(ch_i)$$

检查刻度；只有 residual 呈现系统曲率时才考虑加入 $a_2ch^2$。把刻度应用到完整 histogram 后，变换前后的总计数应保持一致。

### 能量分辨率

由能量刻度的局部斜率把 channel 上的 width 换算为

$$
FWHM(E)=2\sqrt{2\ln2}\left|\frac{dE}{dch}\right|\sigma_{ch}
\approx2.355\left|\frac{dE}{dch}\right|\sigma_{ch}.
$$

HPGe 的 FWHM 随能量变化常用

$$FWHM(E)=\sqrt{A+BE+CE^2}$$

作经验描述；常数项、$E$ 项和 $E^2$ 项分别概括电子学噪声、载流子统计和随能量增长的电荷收集等贡献。

<img src="../calibration_method/width.png" alt="a typical HPGe resolution curve" style="max-width:38%;" />

### Full-energy peak efficiency

把参考活度 $A_0$ 从日期 $t_0$ 修正到测量日期 $t$：

$$
A(t)=A_0\,2^{-(t-t_0)/T_{1/2}}.
$$

本次测量时长远短于两个源的半衰期，因此

$$
\varepsilon(E_\gamma)=
\frac{N_{\mathrm{peak}}}{A(t)P_\gamma t_{\mathrm{live}}}.
$$

若输入量相互独立，其相对误差可由

$$
\left(\frac{u_\varepsilon}{\varepsilon}\right)^2=
\left(\frac{u_N}{N_{\mathrm{peak}}}\right)^2+
\left(\frac{u_A}{A(t)}\right)^2+
\left(\frac{u_P}{P_\gamma}\right)^2+
\left(\frac{u_t}{t_{\mathrm{live}}}\right)^2
$$

估计。同一标准源的活度误差会同时影响该源的全部 efficiency 点，因此这些点之间相关。若尚未修正 dead time、true-coincidence summing、源几何和自吸收，这里得到的是该测量设置下的 apparent full-energy peak efficiency。

HPGe 的 efficiency 在低能端会因探测器端帽、死层和源封装材料的吸收而迅速下降，在中间能区达到最大值后再随能量升高而下降。令

$$u=\ln\!\left(\frac{E}{100\ \mathrm{keV}}\right),$$

本作业采用经验函数

$$
\varepsilon(E)=\exp\!\left[
p_0+p_1u+p_2u^2-p_3\left(\frac{100\ \mathrm{keV}}{E}\right)^3
\right],\qquad p_3\ge0.
$$

前三项描述较高能区的平滑变化，最后一项近似描述低能 photon 的吸收。曲线在 log–log 坐标上显示，并用 residual 检查；本数据最低参考能量为 81 keV，更低能区没有刻度点约束。

若要给出固定能量 $E_0$ 处 fitted efficiency 的误差，令 $q=(100\ \mathrm{keV}/E_0)^3$，则

$$
\mathbf J(E_0)=\varepsilon(E_0)\,(1,\ u,\ u^2,\ -q),\qquad
u_{\mathrm{fit}}[\varepsilon(E_0)]=\sqrt{\mathbf J C\mathbf J^{\mathsf T}},
$$

其中 $C$ 是 efficiency-curve fit 的 parameter covariance matrix。不能只把四个 parameter errors 分别平方相加，因为这些参数通常高度相关。这个 $u_{\mathrm{fit}}$ 只表示给定模型下的拟合曲线误差；未放入该 fit 的源活度相关误差和模型选择误差仍需另行说明。

<img src="../calibration_method/eff.png" alt="a typical HPGe full-energy peak efficiency curve" style="max-width:38%;" />


## 作业要求

### 能量刻度

1. 用 log scale 查看完整能谱。参照标准源能谱和表中的 $E_\gamma$、$P_\gamma$，先找出两个相隔较远且指认可靠的 peak，估计线性刻度关系，再据此寻找其余刻度线。
2. 对参与分析的孤立 peak，先观察左右 sideband 并估计 linear background，再进行 Gaussian + linear background 的 joint fit。报告 centroid、$\sigma$ 和 area，并检查 residual；若出现连续结构，指出初步模型不能描述的区域。
3. 在完整 log-y 能谱上叠加所有参与分析的 signal + background 总函数，并同时画出各自的 fitted background，检查每个函数是否覆盖了正确的 peak 和本底区间。
4. 把 centroid、centroid error 和参考能量填入 `TGraphErrors`。比较线性和二次能量刻度，根据 calibration residual 选择刻度关系。
5. 用选定的刻度关系生成能量谱，并检查变换前后的总计数是否一致。

### 能量分辨率

由同一组局部 fit 的 $\sigma_{ch}$ 计算 FWHM，画出 FWHM–$E_\gamma$ 曲线，用

$$FWHM(E)=\sqrt{A+BE+CE^2}$$

拟合，并画出 $FWHM_{\mathrm{data}}-FWHM_{\mathrm{fit}}$。根据 residual 判断该关系能否描述数据。

### Full-energy peak efficiency（选做）

1. 从 peak fit 的 Gaussian signal 计算 $N_{\mathrm{peak}}$，用拟合返回的完整 covariance matrix 传播统计误差；选择一个孤立 peak，用 sideband subtraction 交叉检查 area。
2. 将两个源的活度修正到测量日期，计算各条 gamma line 的 apparent full-energy peak efficiency。
3. 在 log–log 坐标上用方法部分包含低能吸收项的函数拟合 efficiency–energy 曲线，并画出相对 residual。
4. 选择一个位于数据覆盖范围内的能量 $E_0$，报告 $\varepsilon(E_0)$，并用 efficiency-fit 的完整 parameter covariance matrix 计算 $u_{\mathrm{fit}}[\varepsilon(E_0)]$。


## 实例代码

下面只展开 867.378 keV peak 的 Gaussian + linear background 拟合，用它说明初值、peak area、covariance 和 residual。其余参考线需要按相同步骤独立处理。


<div class="code-language-switch" role="group" aria-label="Code language">
  <span>Code language:</span>
  <button type="button" data-code-language="python" aria-pressed="true">Python / PyROOT</button>
  <button type="button" data-code-language="cpp" aria-pressed="false">ROOT C++</button>
</div>

<style>
.code-language-switch { display:none; gap:.5rem; align-items:center; margin:1rem 0; }
.code-language-switch button { padding:.3rem .8rem; border:1px solid #b8b8b8; border-radius:4px; background:#fff; cursor:pointer; }
.code-language-switch button[aria-pressed="true"] { color:#fff; background:#2f6f9f; border-color:#2f6f9f; }
.pyroot-code-marker { display:none; }
.pyroot-code-marker + .highlight {
  margin:.5rem 0 1rem;
  border:1px solid #d5d5d5;
  border-radius:2px;
  background:#f7f7f7;
}
.pyroot-code-marker + .highlight pre { margin:0; padding:.75rem 1rem; overflow-x:auto; }
.pyroot-code-cell[hidden],
.jp-CodeCell .jp-Cell-inputWrapper[hidden] { display:none !important; }
</style>

<script>
document.addEventListener("DOMContentLoaded", function () {
  const buttons = document.querySelectorAll(".code-language-switch button");
  const pythonCells = Array.from(document.querySelectorAll(".pyroot-code-marker"))
    .map(function (marker) { return marker.closest(".jp-MarkdownCell"); })
    .filter(Boolean);
  pythonCells.forEach(function (cell) { cell.classList.add("pyroot-code-cell"); });
  const cppInputs = document.querySelectorAll(".jp-CodeCell .jp-Cell-inputWrapper");

  function selectLanguage(language) {
    pythonCells.forEach(function (cell) { cell.hidden = language !== "python"; });
    cppInputs.forEach(function (input) { input.hidden = language !== "cpp"; });
    buttons.forEach(function (button) {
      button.setAttribute("aria-pressed", String(button.dataset.codeLanguage === language));
    });
  }

  buttons.forEach(function (button) {
    button.addEventListener("click", function () { selectLanguage(button.dataset.codeLanguage); });
  });
  document.querySelector(".code-language-switch").style.display = "flex";
  selectLanguage("python");
});
</script>

### 读取并查看能谱

先打开文件并取得 `h0`。`%jsroot on`（PyROOT）和 `//%jsroot on`（ROOT C++）开启 notebook 中的交互式图形。完整谱使用 log scale；这一步只改变显示，不改变计数。


<div class="pyroot-code-marker"></div>

```python
import math
import ROOT

%jsroot on
ROOT.gStyle.SetOptStat(0)

# TFile.Open 打开 ROOT 文件；Get 取得其中名为 h0 的 histogram。
input_file = ROOT.TFile.Open("gamma.root", "READ")
h0 = input_file.Get("h0")

c_spectrum = ROOT.TCanvas("c_spectrum_py", "h0", 850, 480)
c_spectrum.SetLogy()
h0.SetTitle("^{152}Eu + ^{133}Ba spectrum;channel;counts / bin")
h0.GetXaxis().SetRangeUser(40, 1300)
h0.SetMinimum(0.5)
h0.Draw("hist")
c_spectrum.Draw()
c_spectrum.SaveAs("standard_source_spectrum.png")
```


In [ ]:
//%jsroot on
#include "TCanvas.h"
#include "TFile.h"
#include "TF1.h"
#include "TFitResultPtr.h"
#include "TGraph.h"
#include "TGraphErrors.h"
#include "TH1.h"
#include "TLegend.h"
#include "TLine.h"
#include "TMath.h"
#include "TStyle.h"
#include <algorithm>
#include <cmath>
#include <iomanip>
#include <iostream>

gStyle->SetOptStat(0);

// TFile::Open 打开 ROOT 文件；Get 取得其中名为 h0 的 histogram。
auto inputFile = TFile::Open("gamma.root", "READ");
auto h0 = dynamic_cast<TH1*>(inputFile->Get("h0"));

auto cSpectrum = new TCanvas("cSpectrum", "h0", 850, 480);
cSpectrum->SetLogy();
h0->SetTitle("^{152}Eu + ^{133}Ba spectrum;channel;counts / bin");
h0->GetXaxis()->SetRangeUser(40, 1300);
h0->SetMinimum(0.5);
h0->Draw("hist");
cSpectrum->Draw();
cSpectrum->SaveAs("standard_source_spectrum.png");


运行后生成前面的标准源测量谱：

<img src="standard_source_spectrum.png" alt="code-generated standard-source spectrum" style="max-width:58%;" />


### 选择 sideband、拟合区间与初始值

多参数 peak model 是非线性拟合，minimizer 从给定初值开始搜索。初值相差过大时，搜索可能进入错误的 local minimum 或不能稳定收敛。下图是原实例代码使用的示意图；它说明的是初值会影响搜索路径，不是要求预先知道最终答案。

<img src="../code/minimum.png" alt="local and global minima in a fit objective" style="max-width:38%;" />

先画出局部数据。sideband 应位于 visible peak 之外，同时靠近 peak；用 sideband 的 linear fit 给出 $b_0$ 和 slope 的初值。peak maximum 减去本底估计给出 $H$ 的初值，peak 位置和可见宽度给出 $\mu$、$\sigma$ 的初值。parameter limits 只用于排除负 peak height、负 peak width 等非物理解。

本例会用到以下几个 ROOT 接口：

| 接口 | 用途 |
| --- | --- |
| `SetParameter(i, value)` | 为第 $i$ 个 parameter 设置初值 |
| `SetParLimits(i, low, high)` | 给 parameter 设置物理允许范围 |
| `Integral(x_1, x_2)` | 计算函数在一个 bin 内的积分，用于 residual |
| `GetCovarianceMatrix()` | 取得同一次 fit 的 parameter covariance matrix |

fit option 为 `LIRSQN`：

| option | 本例中的作用 |
| --- | --- |
| `L` | binned Poisson likelihood |
| `I` | 使用函数在每个 bin 内的积分 |
| `R` | 使用 `TF1` 定义的局部范围 |
| `S` | 返回 `TFitResult`，以读取 covariance matrix |
| `Q` | 不打印完整 minimizer 过程 |
| `N` | 不自动保存或绘制函数；后面自行组织图形 |

各选项的完整定义见 [ROOT `TH1::Fit` documentation](https://root.cern.ch/doc/master/classTH1.html)。


### 867.378 keV：先拟合 sideband

拟合区间取在 867.378 keV peak 附近。先把 visible peak 中心排除，只用左右 sideband 拟合直线。这个结果用于给 joint fit 设置本底初值。


<div class="pyroot-code-marker"></div>

```python
h0.GetXaxis().SetRangeUser(0.0, 2500.0)
xmin867, xmax867 = 756.7, 768.7
x0_867 = 762.7
sidebands867 = ((756.7, 759.7), (765.7, 768.7))

sideband_graph867 = ROOT.TGraphErrors()
point = 0
for low, high in sidebands867:
    for bin_number in range(h0.FindBin(low), h0.FindBin(high) + 1):
        count = h0.GetBinContent(bin_number)
        sideband_graph867.SetPoint(point, h0.GetBinCenter(bin_number), count)
        sideband_graph867.SetPointError(point, 0.0, math.sqrt(max(count, 1.0)))
        point += 1

background_seed867 = ROOT.TF1(
    "background_seed867_py", "[0]+[1]*(x-762.7)", xmin867, xmax867
)
background_seed867.SetParNames("b0", "slope")
background_seed867.SetParameters(5.8e3, 0.0)
background_result867 = sideband_graph867.Fit(background_seed867, "RSQN")

c_background867 = ROOT.TCanvas("c_background867_py", "867 sidebands", 720, 420)
sideband_graph867.SetTitle(
    "867.378 keV: linear fit to sidebands;channel;counts / bin"
)
sideband_graph867.SetMarkerStyle(20)
sideband_graph867.GetXaxis().SetLimits(xmin867, xmax867)
sideband_graph867.SetMinimum(0.0)
sideband_graph867.Draw("AP")
background_seed867.SetLineColor(ROOT.kRed + 1)
background_seed867.Draw("same")
c_background867.Draw()
c_background867.SaveAs("fit_867_background.png")
```


In [ ]:
h0->GetXaxis()->SetRangeUser(0.0, 2500.0);
double xmin867 = 756.7;
double xmax867 = 768.7;
double x0_867 = 762.7;

auto sidebandGraph867 = new TGraphErrors();
int sidebandPoint867 = 0;
const double sidebands867[2][2] = {{756.7, 759.7}, {765.7, 768.7}};
for (const auto& interval : sidebands867) {
    for (int bin = h0->FindBin(interval[0]); bin <= h0->FindBin(interval[1]); ++bin) {
        double count = h0->GetBinContent(bin);
        sidebandGraph867->SetPoint(
            sidebandPoint867, h0->GetBinCenter(bin), count);
        sidebandGraph867->SetPointError(
            sidebandPoint867, 0.0, std::sqrt(std::max(count, 1.0)));
        ++sidebandPoint867;
    }
}

auto backgroundSeed867 = new TF1(
    "backgroundSeed867", "[0]+[1]*(x-762.7)", xmin867, xmax867);
backgroundSeed867->SetParNames("b0", "slope");
backgroundSeed867->SetParameters(5.8e3, 0.0);
TFitResultPtr backgroundResult867 = sidebandGraph867->Fit(
    backgroundSeed867, "RSQN");

auto cBackground867 = new TCanvas("cBackground867", "867 sidebands", 720, 420);
sidebandGraph867->SetTitle(
    "867.378 keV: linear fit to sidebands;channel;counts / bin");
sidebandGraph867->SetMarkerStyle(20);
sidebandGraph867->GetXaxis()->SetLimits(xmin867, xmax867);
sidebandGraph867->SetMinimum(0.0);
sidebandGraph867->Draw("AP");
backgroundSeed867->SetLineColor(kRed + 1);
backgroundSeed867->Draw("same");
cBackground867->Draw();
cBackground867->SaveAs("fit_867_background.png");


### Gaussian + linear background joint fit

用 sideband fit 的结果初始化本底参数，再在完整区间内同时拟合 Gaussian 和 linear background。此时 `b0` 和 `slope` 没有固定，会随 peak 参数一起由数据确定。


<div class="pyroot-code-marker"></div>

```python
model867 = "gaus(0)+[3]+[4]*(x-762.7)"
f867 = ROOT.TF1("f867_py", model867, xmin867, xmax867)
f867.SetParNames("height", "mean", "sigma", "b0", "slope")
f867.SetParameter(0, 4.5e4)                            # peak height
f867.SetParameter(1, 762.7)                            # centroid
f867.SetParameter(2, 0.8)                              # sigma
f867.SetParameter(3, background_seed867.GetParameter(0))  # b0 from sidebands
f867.SetParameter(4, background_seed867.GetParameter(1))  # slope from sidebands
f867.SetParLimits(0, 0.0, 1.0e7)
f867.SetParLimits(1, 760.5, 765.0)
f867.SetParLimits(2, 0.2, 3.0)

result867 = h0.Fit(f867, "LIRSQN")
```


In [ ]:
auto f867 = new TF1(
    "f867", "gaus(0)+[3]+[4]*(x-762.7)", xmin867, xmax867);
f867->SetParNames("height", "mean", "sigma", "b0", "slope");
f867->SetParameter(0, 4.5e4);                              // peak height
f867->SetParameter(1, 762.7);                              // centroid
f867->SetParameter(2, 0.8);                                // sigma
f867->SetParameter(3, backgroundSeed867->GetParameter(0)); // b0 from sidebands
f867->SetParameter(4, backgroundSeed867->GetParameter(1)); // slope from sidebands
f867->SetParLimits(0, 0.0, 1.0e7);
f867->SetParLimits(1, 760.5, 765.0);
f867->SetParLimits(2, 0.2, 3.0);

TFitResultPtr result867 = h0->Fit(f867, "LIRSQN");


### 读取 centroid、$\sigma$ 和 area

`GetParameter(1)` 和 `GetParError(1)` 分别给出 centroid 及其拟合误差；`GetParameter(2)` 给出 $\sigma$。注意 centroid error 不是 $\sigma$。Gaussian signal 的 $H$ 和 $\sigma$ 共同决定 area。


<div class="pyroot-code-marker"></div>

```python
height867 = f867.GetParameter(0)
mean867 = f867.GetParameter(1)
sigma867 = abs(f867.GetParameter(2))
area_factor = math.sqrt(2.0 * math.pi) / h0.GetBinWidth(1)
area867 = height867 * sigma867 * area_factor
```


In [ ]:
double height867 = f867->GetParameter(0);
double mean867 = f867->GetParameter(1);
double sigma867 = std::abs(f867->GetParameter(2));
double areaFactor = std::sqrt(2.0 * TMath::Pi()) / h0->GetBinWidth(1);
double area867 = height867 * sigma867 * areaFactor;


#### 使用 covariance 计算 area error

`GetCovarianceMatrix()` 返回同一次 fit 的 parameter covariance matrix。常见错误是只传播 height error 与 $\sigma$ error，却漏掉二者的 covariance；这里必须保留交叉项。`CovMatrixStatus()` 用来检查矩阵质量；ROOT 返回 3 表示 full, accurate covariance matrix。


<div class="pyroot-code-marker"></div>

```python
cov867 = result867.GetCovarianceMatrix()
area_variance867 = (
    (sigma867 * area_factor)**2 * cov867[0][0]
    + (height867 * area_factor)**2 * cov867[2][2]
    + 2.0 * height867 * sigma867 * area_factor**2 * cov867[0][2]
)
area_error867 = math.sqrt(max(area_variance867, 0.0))

print("867.378 keV example")
print(f"mean   = {mean867:.4f} +/- {f867.GetParError(1):.4f}")
print(f"sigma  = {sigma867:.4f} +/- {f867.GetParError(2):.4f}")
print(f"N_peak = {area867:.0f} +/- {area_error867:.0f}")
print(f"fit status = {int(result867)}")
print(f"covariance status = {result867.CovMatrixStatus()}")
```


In [ ]:
auto cov867 = result867->GetCovarianceMatrix();
double areaVariance867 =
    std::pow(sigma867 * areaFactor, 2) * cov867(0, 0)
    + std::pow(height867 * areaFactor, 2) * cov867(2, 2)
    + 2.0 * height867 * sigma867 * areaFactor * areaFactor * cov867(0, 2);
double areaError867 = std::sqrt(std::max(areaVariance867, 0.0));

std::cout << "867.378 keV example\n"
          << "mean   = " << std::fixed << std::setprecision(4)
          << mean867 << " +/- " << f867->GetParError(1) << "\n"
          << "sigma  = " << sigma867 << " +/- " << f867->GetParError(2) << "\n"
          << "N_peak = " << std::setprecision(0)
          << area867 << " +/- " << areaError867 << "\n"
          << "fit status = " << static_cast<int>(result867) << "\n"
          << "covariance status = " << result867->CovMatrixStatus() << "\n";


### 计算并绘制 residual

逐 bin 计算模型期望和 Pearson residual。因为拟合使用了 `I`，这里也用 `Integral` 对函数作 bin integral，再除以 bin width 得到该 bin 的期望高度。


<div class="pyroot-code-marker"></div>

```python
residual867 = ROOT.TGraph()
for point, bin_number in enumerate(
    range(h0.FindBin(xmin867), h0.FindBin(xmax867) + 1)
):
    low = h0.GetBinLowEdge(bin_number)
    width = h0.GetBinWidth(bin_number)
    expected = f867.Integral(low, low + width) / width
    observed = h0.GetBinContent(bin_number)
    residual867.SetPoint(
        point, h0.GetBinCenter(bin_number),
        (observed - expected) / math.sqrt(expected)
    )
```


In [ ]:
auto residual867 = new TGraph();
int point867 = 0;
for (int bin = h0->FindBin(xmin867); bin <= h0->FindBin(xmax867); ++bin) {
    double low = h0->GetBinLowEdge(bin);
    double width = h0->GetBinWidth(bin);
    double expected = f867->Integral(low, low + width) / width;
    double observed = h0->GetBinContent(bin);
    residual867->SetPoint(
        point867++, h0->GetBinCenter(bin),
        (observed - expected) / std::sqrt(expected));
}


将局部 peak、signal + background 总函数和 fitted background 画在上方，residual 画在下方。局部 peak 图采用 linear y；显示用的 histogram 是 `h0` 的 clone，不会改变原始能谱。


<div class="pyroot-code-marker"></div>

```python
c867 = ROOT.TCanvas("c867_py", "867.378 keV example", 720, 650)
c867.Divide(1, 2)
c867.cd(1)
h867_view = h0.Clone("h867_view_py")
h867_view.GetXaxis().SetRangeUser(xmin867, xmax867)
h867_view.SetTitle("867.378 keV example;channel;counts / bin")
h867_view.SetMinimum(0.0)
h867_view.Draw("E")
f867.SetLineColor(ROOT.kBlue + 1)
f867.Draw("same")
background867 = ROOT.TF1(
    "background867_py", "[0]+[1]*(x-762.7)", xmin867, xmax867
)
background867.SetParameters(f867.GetParameter(3), f867.GetParameter(4))
background867.SetLineColor(ROOT.kRed + 1)
background867.SetLineStyle(2)
background867.Draw("same")
legend867 = ROOT.TLegend(0.55, 0.70, 0.88, 0.88)
legend867.AddEntry(f867, "signal + background", "l")
legend867.AddEntry(background867, "fitted background", "l")
legend867.Draw()

c867.cd(2)
residual867.SetTitle("Fit residual;channel;(n-#nu)/#sqrt{#nu}")
residual867.SetMarkerStyle(20)
residual867.Draw("AP")
zero867 = ROOT.TLine(xmin867, 0.0, xmax867, 0.0)
zero867.SetLineStyle(2)
zero867.Draw()
c867.Draw()
c867.SaveAs("fit_867_linear.png")
```


In [ ]:
auto c867 = new TCanvas("c867", "867.378 keV example", 720, 650);
c867->Divide(1, 2);
c867->cd(1);
auto h867View = static_cast<TH1*>(h0->Clone("h867View"));
h867View->GetXaxis()->SetRangeUser(xmin867, xmax867);
h867View->SetTitle("867.378 keV example;channel;counts / bin");
h867View->SetMinimum(0.0);
h867View->Draw("E");
f867->SetLineColor(kBlue + 1);
f867->Draw("same");
auto background867 = new TF1(
    "background867", "[0]+[1]*(x-762.7)", xmin867, xmax867);
background867->SetParameters(f867->GetParameter(3), f867->GetParameter(4));
background867->SetLineColor(kRed + 1);
background867->SetLineStyle(2);
background867->Draw("same");
auto legend867 = new TLegend(0.55, 0.70, 0.88, 0.88);
legend867->AddEntry(f867, "signal + background", "l");
legend867->AddEntry(background867, "fitted background", "l");
legend867->Draw();

c867->cd(2);
residual867->SetTitle("Fit residual;channel;(n-#nu)/#sqrt{#nu}");
residual867->SetMarkerStyle(20);
residual867->Draw("AP");
auto zero867 = new TLine(xmin867, 0.0, xmax867, 0.0);
zero867->SetLineStyle(2);
zero867->Draw();
c867->Draw();
c867->SaveAs("fit_867_linear.png");


运行后得到 sideband background、867.378 keV peak 的 joint fit 与 residual：

<div class="peak-example-grid">
  <figure>
    <img src="fit_867_background.png" alt="code-generated sideband background fit" />
    <figcaption>只使用左右 sideband 的 linear fit。</figcaption>
  </figure>
  <figure>
    <img src="fit_867_linear.png" alt="code-generated linear-background peak fit" />
    <figcaption>完整局部区间内的 joint fit 与 residual。</figcaption>
  </figure>
</div>


## 参考结果

以下结果用于完成作业后的核对。867.378 keV 示例给出：

| gamma line | local model | centroid (channel) | $\sigma$ (channel) | $N_{\mathrm{peak}}$ |
| --- | --- | ---: | ---: | ---: |
| 867.378 keV | Gaussian + linear background | $762.6016\pm0.0015$ | $0.8180\pm0.0014$ | $(4.7918\pm0.0084)\times10^5$ |

fit status 为 0，covariance status 为 3。residual 中仍可见小的系统结构，说明 linear background 是初步模型；fit status 不能代替 residual 检查。

### 全部局部拟合

下面把所有用于能量刻度、FWHM 和 efficiency 的局部拟合叠加在完整 log-y 能谱上。蓝色实线是各峰的 signal + background 总函数，红色虚线是对应的 fitted linear background；每组函数只画在自己的拟合区间内。

<img src="../code/all_peak_fits.png" alt="all local signal-plus-linear-background fits overlaid on the gamma spectrum" style="max-width:68%;" />

使用全部刻度线后，一次能量刻度给出

$$E_\gamma\;(\mathrm{keV})\approx-39.924+1.189801\,ch,$$

线性刻度的最大绝对 calibration residual 约为 0.09 keV；二次函数改善很小，因此参考结果仍采用线性刻度。

<style>
.result-grid { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:1rem; margin:1rem 0 1.5rem; }
.result-grid figure { margin:0; padding:.6rem; border:1px solid #ddd; background:#fff; }
.result-grid img { display:block; width:100%; height:auto; }
.result-grid figcaption { margin-top:.5rem; font-size:.92rem; line-height:1.4; }
@media (max-width:720px) { .result-grid { grid-template-columns:1fr; } }
</style>

### 能量刻度

<div class="result-grid">
  <figure>
    <img src="reference_calibrated_spectrum.png" alt="calibrated gamma spectrum" />
    <figcaption>将最终刻度关系应用于原始 histogram 后得到的能量谱；变换前后总计数一致。</figcaption>
  </figure>
  <figure>
    <img src="reference_energy_calibration.png" alt="energy calibration and residuals" />
    <figcaption>线性与二次刻度关系及其 calibration residual。残差决定是否需要二次项。</figcaption>
  </figure>
</div>

500 keV 处的 fitted apparent efficiency 为 $10.5668\%\pm0.0258\%$；这里的误差只来自 efficiency-fit parameter covariance。

### 峰宽与效率

<div class="result-grid">
  <figure>
    <img src="reference_fwhm.png" alt="FWHM versus energy and residuals" />
    <figcaption>FWHM–$E_\gamma$ 曲线及 residual；这里使用 $\sqrt{A+BE+CE^2}$。</figcaption>
  </figure>
  <figure>
    <img src="reference_efficiency.png" alt="apparent full-energy peak efficiency and residuals" />
    <figcaption>Apparent full-energy peak efficiency 及相对 residual，横纵坐标均为 log scale；拟合函数包含低能吸收项，81 keV 以下的曲线是未被数据约束的外推。</figcaption>
  </figure>
</div>
